In [ ]:
import requests

res = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "gemma4:e2b",
        "prompt": "Im calling an api to a local host. Do you work with file paths?",
        "stream": False,
    },
    timeout=60,
)
res.raise_for_status()

data = res.json()
print(data["response"])


As a Large Language Model, I don't directly interact with your local file system or execute the API calls myself. I don't "work" with file paths in the sense of reading files or navigating your computer.

**However, I can absolutely help you with the logic, code, and structure related to file paths and API calls.**

If you are trying to call an API to a local host using a file path, I can assist you with things like:

1.  **Constructing the URL:** How to correctly format a local path into a network address (e.g., using `file://` or specific protocols).
2.  **Programming Logic:** Writing the Python, JavaScript, or other language code required to read a file and send its contents to a local server.
3.  **Error Handling:** Troubleshooting common issues related to path access or local networking.

**To give you the best help, please tell me:**

*   **What programming language are you using?** (e.g., Python, Node.js)
*   **What is the goal of the API call?** (e.g., Are you trying to upload 

# Karma scoring

For each of the top 100 characters, send their biography + their list of affiliated targets to local Gemma and ask for a 1–10 karma score per target.


In [4]:
import requests
import pandas as pd
import json
import re
import csv
from tqdm.auto import tqdm

top = pd.read_csv("top100_outgoing.csv")
bios = pd.read_csv("characters_bio.csv")
bio_lookup = dict(zip(bios["ID"], bios["bio"].fillna("")))
name_lookup = dict(zip(bios["ID"], bios["name"].fillna(bios["ID"])))
print(len(top), "sources")


100 sources


In [5]:
def score(source_id, target_ids):
    bio = bio_lookup.get(source_id, "")
    name = name_lookup.get(source_id, source_id)

    prompt = f"""You are analyzing the biography of {name} from A Song of Ice and Fire.
For every character ID listed, give a karma score from 1 to 10 reflecting how {name} feels about them, using the bio AND your general knowledge of the books.

1 = extremely hostile (sworn enemy, killer, betrayer)
10 = extremely friendly (closest ally, beloved family, sworn friend)

Biography:
{bio}

Score these character IDs (use them EXACTLY as JSON keys):
{', '.join(target_ids)}

Return ONLY a JSON object like {{"ID_one": 7, "ID_two": 2}}. No prose, no code fences."""

    res = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "gemma4:e4b",
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0.4, "num_ctx": 16384, "num_predict": 6000},
        },
        timeout=900,
    )
    res.raise_for_status()
    text = res.json()["response"].strip()
    # strip code fences if present
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text)
    m = re.search(r"\{.*\}", text, re.DOTALL)
    return json.loads(m.group(0)) if m else {}


In [6]:
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

with open("karma_edges.csv", "w", newline="") as f:
    csv.writer(f).writerow(["source_id", "target_id", "karma_score"])

write_lock = threading.Lock()

def process(row):
    sid = row.ID
    targets = [t for t in (row.outgoing_edges or "").split(";") if t]
    if not targets:
        return sid, 0
    try:
        scores = score(sid, targets)
    except Exception as e:
        print(f"{sid}: {e}")
        return sid, 0
    rows = []
    for tid, s in scores.items():
        try:
            s = max(1, min(10, int(s)))
        except (TypeError, ValueError):
            continue
        if tid in targets:
            rows.append([sid, tid, s])
    with write_lock, open("karma_edges.csv", "a", newline="") as f:
        csv.writer(f).writerows(rows)
    return sid, len(rows)

jobs = list(top.itertuples(index=False))
with ThreadPoolExecutor(max_workers=4) as ex:
    futures = [ex.submit(process, row) for row in jobs]
    for f in tqdm(as_completed(futures), total=len(futures)):
        f.result()


  3%|▎         | 3/100 [05:58<3:10:35, 117.89s/it]

Tyrion_Lannister: Expecting ',' delimiter: line 1 column 3649 (char 3648)


  4%|▍         | 4/100 [07:17<2:43:43, 102.33s/it]

Jaime_Lannister: Expecting ',' delimiter: line 1 column 2245 (char 2244)


 16%|█▌        | 16/100 [33:43<2:43:05, 116.50s/it]

...

 47%|████▋     | 47/100 [1:24:32<1:05:52, 74.57s/it]

Daemon_Velaryon_(son_of_Aethan): Expecting ',' delimiter: line 1 column 2282 (char 2281)


 49%|████▉     | 49/100 [1:26:05<51:09, 60.19s/it]  

Otto_Hightower: Expecting ',' delimiter: line 1 column 2248 (char 2247)


100%|██████████| 100/100 [2:50:03<00:00, 102.04s/it] 


## Quick look

In [7]:
edges = pd.read_csv("karma_edges.csv")
print(len(edges), "edges,", edges["source_id"].nunique(), "sources")
print(edges["karma_score"].value_counts().sort_index())


9724 edges, 87 sources
karma_score
1      513
2      707
3     1543
4      807
5     3738
6     1035
7      753
8      341
9      190
10      97
Name: count, dtype: int64


In [8]:
edges["source"] = edges["source_id"].map(name_lookup)
edges["target"] = edges["target_id"].map(name_lookup)
print("Friendliest:")
print(edges.sort_values("karma_score", ascending=False).head(10)[["source","target","karma_score"]].to_string(index=False))
print("\nMost hostile:")
print(edges.sort_values("karma_score").head(10)[["source","target","karma_score"]].to_string(index=False))


Friendliest:
            source                 target  karma_score
       Walder Frey              Lord Frey           10
     Catelyn Stark           Eddard Stark           10
 Daenaera Velaryon        Daeron Velaryon           10
     Balon Greyjoy        Balon V Greyjoy           10
     Brandon Stark         Brynden Rivers           10
Rhaenyra Targaryen       Lucerys Velaryon           10
Rhaenyra Targaryen      Jacaerys Velaryon           10
Baelor I Targaryen           High Sparrow           10
Baelor I Targaryen  High Septon (fat one)           10
Baelor I Targaryen High Septon (Baelor I)           10

Most hostile:
           source             target  karma_score
 Ormund Baratheon   Valarr Targaryen            1
  Renly Baratheon Aerys II Targaryen            1
      Ramsay Snow           Jon Snow            1
 Qarlton Chelsted        Edmyn Tully            1
 Qarlton Chelsted     Orys Baratheon            1
Viserys Targaryen   Shaera Targaryen            1
 Qarlton Chelsted

## Re-run missing sources (chunked)

The first pass dropped 13 high-out-degree sources because the model returned malformed JSON when the target list got long. Here we identify which `top100` sources are missing from `karma_edges.csv` and re-score them in **chunks of 40 targets** so each JSON response stays short and parses cleanly. Results are **appended** to `karma_edges.csv`; chunk-level errors go to `karma_failures.csv`.

In [ ]:
import os
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

CHUNK = 40  # targets per LLM call — keeps JSON response well under the parse-failure threshold

# Which top100 sources are currently missing (0 edges) in karma_edges.csv?
existing = pd.read_csv("karma_edges.csv")
have = set(existing["source_id"].unique())
missing_sources = [sid for sid in top["ID"] if sid not in have]
print(f"Missing sources to re-run: {len(missing_sources)}")
for s in missing_sources:
    print(" -", s)

# Make sure failures file has a header
if not os.path.exists("karma_failures.csv") or os.path.getsize("karma_failures.csv") == 0:
    with open("karma_failures.csv", "w", newline="") as f:
        csv.writer(f).writerow(["source_id", "chunk_index", "reason", "raw_response"])

# We'll call score() per chunk. score() returns {} on no match, so for chunked work
# we want raw responses on failure — wrap in a chunk-aware helper.
def score_chunk(source_id, target_chunk):
    bio = bio_lookup.get(source_id, "")
    name = name_lookup.get(source_id, source_id)
    prompt = f"""You are analyzing the biography of {name} from A Song of Ice and Fire.
For every character ID listed, give a karma score from 1 to 10 reflecting how {name} feels about them, using the bio AND your general knowledge of the books.

1 = extremely hostile (sworn enemy, killer, betrayer)
10 = extremely friendly (closest ally, beloved family, sworn friend)

Biography:
{bio}

Score these character IDs (use them EXACTLY as JSON keys):
{', '.join(target_chunk)}

Return ONLY a JSON object like {{"ID_one": 7, "ID_two": 2}}. No prose, no code fences."""
    res = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "gemma4:e4b",
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0.4, "num_ctx": 16384, "num_predict": 4000},
        },
        timeout=900,
    )
    res.raise_for_status()
    text = res.json()["response"].strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text)
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        raise ValueError("no JSON object found")
    return json.loads(m.group(0)), text

write_lock = threading.Lock()

def process_missing(sid):
    row = top.loc[top["ID"] == sid].iloc[0]
    targets = [t for t in (row["outgoing_edges"] or "").split(";") if t]
    if not targets:
        return sid, 0, 0
    chunks = [targets[i:i + CHUNK] for i in range(0, len(targets), CHUNK)]
    target_set = set(targets)
    total_rows = 0
    failed_chunks = 0
    for idx, chunk in enumerate(chunks):
        try:
            scores, _ = score_chunk(sid, chunk)
        except Exception as e:
            failed_chunks += 1
            with write_lock, open("karma_failures.csv", "a", newline="") as f:
                csv.writer(f).writerow([sid, idx, str(e), ""])
            continue
        rows = []
        for tid, s in scores.items():
            try:
                s = max(1, min(10, int(s)))
            except (TypeError, ValueError):
                continue
            if tid in target_set:
                rows.append([sid, tid, s])
        with write_lock, open("karma_edges.csv", "a", newline="") as f:
            csv.writer(f).writerows(rows)
        total_rows += len(rows)
    return sid, total_rows, failed_chunks

with ThreadPoolExecutor(max_workers=4) as ex:
    futures = {ex.submit(process_missing, sid): sid for sid in missing_sources}
    for f in tqdm(as_completed(futures), total=len(futures)):
        sid, n, fails = f.result()
        print(f"{sid}: +{n} edges ({fails} failed chunks)")
